# Visualize CoolRun AI Inputs

This notebook visualizes the Vienna orthofoto image, detected tree GeoJSON, and UTCI test polygon before running the final app.

It does not call the Infrared SDK.

If Step 04 produced real UTCI `.npy` grids, this notebook also visualizes the cooling effect.

In [ ]:
# Import the libraries we need.
# sys lets us add the project folder to Python's import path.
# Path helps us build file paths that work on different operating systems.
import json
import sys
from pathlib import Path

In [ ]:
# Find the project root folder.
# If this notebook is opened from the notebooks/ folder, the project root is one level up.
# If it is opened from the project root, the current folder is already the project root.
current_dir = Path.cwd()

if (current_dir / ".env").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent

# Add the project root to Python's import path.
# This lets the notebook import code from backend/app.
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
# Import reusable helpers.
# create_bbox_polygon gives us the same Vienna orthofoto bounds polygon used by notebook 04.
# input_maps contains plotting and map export functions.
import importlib

from backend.app.imagery.vienna_orthofoto import create_bbox_polygon
import backend.app.visualization.input_maps as input_maps

# Reload the module so Jupyter picks up recent edits without requiring a kernel restart.
input_maps = importlib.reload(input_maps)

count_point_features = input_maps.count_point_features
load_geojson = input_maps.load_geojson
load_json_if_exists = input_maps.load_json_if_exists
save_cooling_effect_visualization = input_maps.save_cooling_effect_visualization
save_cooling_effect_tree_overlay = input_maps.save_cooling_effect_tree_overlay
save_interactive_input_map = input_maps.save_interactive_input_map
save_static_input_visualization = input_maps.save_static_input_visualization
summarize_tree_utci_overlap = input_maps.summarize_tree_utci_overlap
plot_utci_grids = input_maps.plot_utci_grids

In [ ]:
# Define input and output file paths.
image_path = project_root / "data" / "vienna_orthofoto_test.png"
metadata_path = project_root / "data" / "vienna_orthofoto_test_metadata.json"
tree_geojson_path = project_root / "outputs" / "detected_trees.geojson"
infrared_points_geojson_path = project_root / "outputs" / "infrared_tree_points.geojson"
canopy_geojson_path = project_root / "outputs" / "infrared_tree_canopies.geojson"
utci_summary_path = project_root / "outputs" / "utci_summary.json"
utci_without_trees_path = project_root / "outputs" / "utci_without_trees.npy"
utci_with_trees_path = project_root / "outputs" / "utci_with_trees.npy"
static_output_path = project_root / "outputs" / "input_tree_visualization.png"
interactive_output_path = project_root / "outputs" / "input_tree_map.html"
cooling_effect_output_path = project_root / "outputs" / "utci_cooling_effect.png"
cooling_tree_overlay_output_path = project_root / "outputs" / "utci_cooling_effect_tree_overlay.png"
tree_utci_overlap_summary_path = project_root / "outputs" / "tree_utci_overlap_summary.json"

# Stop early with clear messages if previous steps have not been run.
if not image_path.exists():
    raise FileNotFoundError(
        f"Missing Vienna orthofoto image: {image_path}. Run notebooks/01b_vienna_orthofoto_test.ipynb first."
    )

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Missing Vienna orthofoto metadata: {metadata_path}. Run notebooks/01b_vienna_orthofoto_test.ipynb first."
    )

if not tree_geojson_path.exists():
    raise FileNotFoundError(
        f"Missing tree GeoJSON: {tree_geojson_path}. Run notebooks/03_tree_geojson_conversion.ipynb first."
    )

In [ ]:
# Load the Vienna orthofoto image, georeferencing metadata, and tree GeoJSON.
from PIL import Image

orthofoto_image = Image.open(image_path)
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
default_center_lat = metadata["center"]["lat"]
default_center_lon = metadata["center"]["lon"]
default_zoom = metadata["zoom"]
bbox = metadata["bbox"]
image_bbox = (bbox["min_x"], bbox["min_y"], bbox["max_x"], bbox["max_y"])
tree_geojson = load_geojson(str(tree_geojson_path))
tree_metadata = tree_geojson.get("metadata", {})
if tree_metadata.get("center") != metadata.get("center") or tree_metadata.get("bbox") != metadata.get("bbox"):
    raise ValueError(
        "Detected tree GeoJSON metadata does not match the current Vienna orthofoto metadata. "
        "Rerun notebooks 02 and 03 after changing notebook 01b coordinates."
    )

infrared_points_geojson = load_json_if_exists(str(infrared_points_geojson_path))
canopy_geojson = load_json_if_exists(str(canopy_geojson_path))

# Create the same Vienna orthofoto bounds polygon used by notebook 04.
test_polygon = create_bbox_polygon(metadata["bbox_lonlat"])

tree_count = count_point_features(tree_geojson)
canopy_radii = [
    feature.get("properties", {}).get("canopy_radius_m")
    for feature in tree_geojson.get("features", [])
    if feature.get("properties", {}).get("canopy_radius_m") is not None
]

print(f"Vienna orthofoto size: {orthofoto_image.size[0]} x {orthofoto_image.size[1]} pixels")
print(f"Default Vienna center: lat={default_center_lat}, lon={default_center_lon}")
print(f"Number of detected trees: {tree_count}")
if infrared_points_geojson is None:
    print("No Infrared point-tree payload found yet. Run notebook 04 to create it.")
else:
    print(f"Infrared point-tree payload features: {len(infrared_points_geojson.get('features', []))}")
if canopy_geojson is None:
    print("No canopy polygon visualization GeoJSON found yet. Run notebook 04 to create it.")
else:
    print(f"Canopy polygon visualization features: {len(canopy_geojson.get('features', []))}")
if canopy_radii:
    print(f"Average estimated canopy radius: {sum(canopy_radii) / len(canopy_radii):.2f} m")
    print(f"Max estimated canopy radius: {max(canopy_radii):.2f} m")

In [ ]:
# Load UTCI summary if Step 04 has already saved one.
# This does not run Infrared; it only reads files from outputs/.
utci_summary = load_json_if_exists(str(utci_summary_path))

if utci_summary is None:
    print("No UTCI summary found yet. Run notebook 04 if you want UTCI output visualization.")
else:
    summary_metadata = utci_summary.get("input_metadata")
    if summary_metadata is None:
        raise ValueError(
            "UTCI summary is missing input_metadata. Rerun notebook 04 so UTCI outputs match the current orthofoto."
        )

    if (
        summary_metadata.get("center") != metadata.get("center")
        or summary_metadata.get("bbox") != metadata.get("bbox")
    ):
        raise ValueError(
            "UTCI summary metadata does not match the current Vienna orthofoto metadata. "
            "Rerun notebook 04 after changing notebook 01b coordinates."
        )

    print("Loaded UTCI summary:")
    print(f"  status: {utci_summary.get('status')}")
    print(f"  is_placeholder: {utci_summary.get('is_placeholder')}")
    print(f"  vegetation without: {utci_summary.get('vegetation_feature_count_without')}")
    print(f"  vegetation with: {utci_summary.get('vegetation_feature_count_with')}")
    print(f"  mean UTCI without: {utci_summary.get('mean_utci_without')}")
    print(f"  mean UTCI with: {utci_summary.get('mean_utci_with')}")
    print(f"  mean cooling effect: {utci_summary.get('mean_cooling_effect')}")
    print(f"  max cooling effect: {utci_summary.get('max_cooling_effect')}")
    print(f"  vegetation geometry: {utci_summary.get('vegetation_geometry')}")

    if utci_summary.get("vegetation_geometry") != "points":
        raise RuntimeError(
            "Current UTCI outputs were not produced with Infrared point-tree vegetation. "
            "Rerun notebook 04 with vegetation_geometry_for_utci='points'."
        )

    if (
        utci_summary.get("status") == "completed"
        and utci_summary.get("vegetation_feature_count_with", 0) > 0
        and (
            utci_summary.get("max_abs_utci_difference") == 0
            or (
                utci_summary.get("max_abs_utci_difference") is None
                and utci_summary.get("max_cooling_effect") == 0
            )
        )
    ):
        raise RuntimeError(
            "UTCI outputs contain detected vegetation but with-tree and without-tree grids are identical. "
            "Rerun notebook 04 after confirming vegetation_geometry_for_utci='points'."
        )

In [ ]:
# Plot the Vienna orthofoto, overlay detected tree points, and draw the polygon boundary.
# This saves a static PNG that can be used in reports or the hackathon demo.
saved_static_path = save_static_input_visualization(
    image_path=str(image_path),
    tree_geojson=tree_geojson,
    polygon=test_polygon,
    output_path=str(static_output_path),
    center_lon=default_center_lon,
    center_lat=default_center_lat,
    zoom=default_zoom,
    canopy_geojson=canopy_geojson,
    image_bbox=image_bbox,
)

print(f"Saved static visualization to: {saved_static_path}")

In [ ]:
# Display the saved static visualization inside the notebook.
import matplotlib.pyplot as plt

visualization_image = Image.open(saved_static_path)

plt.figure(figsize=(9, 9))
plt.imshow(visualization_image)
plt.axis("off")
plt.show()

In [ ]:
# Create an interactive Folium map with the polygon and detected tree points.
# This uses web map tiles and does not call the Infrared SDK.
# If Folium is not installed in this notebook kernel, skip only this HTML map.
try:
    saved_interactive_path = save_interactive_input_map(
        tree_geojson=tree_geojson,
        polygon=test_polygon,
        output_path=str(interactive_output_path),
        center_lon=default_center_lon,
        center_lat=default_center_lat,
        canopy_geojson=canopy_geojson,
    )
    print(f"Saved interactive map to: {saved_interactive_path}")
except ModuleNotFoundError as exc:
    if exc.name == "folium":
        print("Folium is not installed in this notebook kernel.")
        print("Run this in a notebook cell, then rerun this cell: %pip install folium")
    else:
        raise

In [ ]:
# If real UTCI grids exist, visualize the cooling effect.
# Cooling effect is calculated as: UTCI without trees - UTCI with trees.
has_real_utci = (
    utci_summary is not None
    and not utci_summary.get("is_placeholder", True)
    and utci_without_trees_path.exists()
    and utci_with_trees_path.exists()
)

if has_real_utci:
    saved_cooling_path = save_cooling_effect_visualization(
        without_trees_npy=str(utci_without_trees_path),
        with_trees_npy=str(utci_with_trees_path),
        output_path=str(cooling_effect_output_path),
    )
    print(f"Saved UTCI cooling effect visualization to: {saved_cooling_path}")

    saved_overlay_path = save_cooling_effect_tree_overlay(
        tree_geojson=tree_geojson,
        without_trees_npy=str(utci_without_trees_path),
        with_trees_npy=str(utci_with_trees_path),
        output_path=str(cooling_tree_overlay_output_path),
        image_bbox=image_bbox,
    )
    print(f"Saved UTCI cooling/tree overlay to: {saved_overlay_path}")

    tree_utci_overlap_summary = summarize_tree_utci_overlap(
        tree_geojson=tree_geojson,
        without_trees_npy=str(utci_without_trees_path),
        with_trees_npy=str(utci_with_trees_path),
        image_bbox=image_bbox,
    )
    tree_utci_overlap_summary_path.write_text(
        json.dumps(tree_utci_overlap_summary, indent=2),
        encoding="utf-8",
    )
    print(f"Saved tree/UTCI overlap summary to: {tree_utci_overlap_summary_path}")
    print(
        "Tree centers on valid UTCI cells: "
        f"{tree_utci_overlap_summary['tree_centers_on_valid_utci_cells']} / {tree_utci_overlap_summary['tree_count']}"
    )
    print(
        "Tree canopy windows with positive cooling: "
        f"{tree_utci_overlap_summary['tree_canopy_windows_with_positive_cooling']} / {tree_utci_overlap_summary['tree_count']}"
    )
else:
    print("No real UTCI grids available. Skipping cooling-effect visualization.")

In [ ]:
# Plot the UTCI arrays inline if Step 04 produced real outputs.
if has_real_utci:
    plot_utci_grids(
        str(utci_without_trees_path),
        str(utci_with_trees_path),
    )

    if cooling_tree_overlay_output_path.exists():
        overlay_image = Image.open(cooling_tree_overlay_output_path)
        plt.figure(figsize=(8, 7))
        plt.imshow(overlay_image)
        plt.axis("off")
        plt.show()
else:
    print("No real UTCI grids available for inline plotting.")